# Cell 1 - Import required libraries

In [1]:
import io
import os
import json
import pandas as pd
from urllib.parse import urlparse
from fastai.learner import load_learner
from fastai.vision import *
from fastai.vision.core import *
from io import BytesIO
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from cmath import sqrt
from typing import List, Dict, ByteString
import base64
import EnclaveSDK
from EnclaveSDK import File, Report, LogData

# Cell 2 - Setup Enclave configuration

In [2]:
enclave_url = os.getenv("ENCLAVE_URL", "https://enclaveapi.escrow.beekeeperai.com")
configuration = EnclaveSDK.Configuration(enclave_url)

# Use the SAS_URL environment variables to use the Data API in the Sandbox, otherwise default to None
sas_url = os.getenv("SAS_URL", None)
if sas_url:
    sas_url = base64.b64encode(sas_url.encode()).decode()

api_client = EnclaveSDK.ApiClient(configuration)

# Cell 3 - Define API helper functions

In [3]:
def get_file_list(sas_url=None) -> List[File]:
    api_instance = EnclaveSDK.DataApi(api_client)
    api_response = api_instance.api_v1_data_files_get(sas_url=sas_url)
    return api_response.files

def download_file(file_name: str, sas_url=None) -> ByteString:
    api_instance = EnclaveSDK.DataApi(api_client)
    content = api_instance.api_v1_data_file_get(file_name, sas_url=sas_url)
    return content

def post_log(log: Dict) -> Dict:
    api_instance = EnclaveSDK.LogApi(api_client)
    log = LogData.from_dict(log)
    api_response = api_instance.api_v1_log_post(log)
    return api_response

def post_report(finalReport: Dict) -> Dict:
    api_instance = EnclaveSDK.ReportApi(api_client)
    
    if os.path.exists("schema.json"):
        with open("schema.json", "r") as schema:
            finalReport['json_schema'] = json.load(schema)
    
    report = Report.from_dict(finalReport)
    api_response = api_instance.api_v1_report_post(report)
    return api_response

# Cell 4 - Define model helper functions

In [4]:
def label_func(x): 
    return x.parent.name 

def getPred(model, file_content):
    uploadedImage = load_image(BytesIO(file_content)).reshape(256,256)
    
    if uploadedImage.mode == 'RGBA':
        uploadedImage = PILImage(uploadedImage.convert('RGB'))
    pred_class,pred_idx,outputs = model.predict(PILImage(uploadedImage))
    
    return pred_class

# Cell 5 - Define report generation function

In [5]:
def generateReport(results):
    reportJSON = {}
    output = io.StringIO()
    
    resultsDf = pd.DataFrame(results)
    if resultsDf.empty:
        post_log({"message": "No results to generate a report", "status": "Failed"})
    
    y_test = resultsDf['actual'].tolist()
    y_pred = resultsDf['prediction'].tolist()
    
    try:
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    except ValueError as e:
        post_log({"message": f"Not enough data to calculate confusion matrix: {str(e)}", "status": "Failed"})
        exit(1)
        
    n = len(y_test)

    try:
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        accuracyCI = 1.96*(sqrt(accuracy)-(1-accuracy))/n
        reportJSON['accuracy'] = {'value': accuracy, 'CI': round(abs(accuracyCI),4), 'n': n}

        specificity = tn/(tn+fp)
        specificityCI = 1.96*(sqrt(specificity)-(1-specificity))/n
        reportJSON['specificity'] = {'value': specificity, 'CI': round(abs(specificityCI),4), 'n': n}

        sensitivity = recall_score(y_test, y_pred, average='weighted')
        sensitivityCI = 1.96*(sqrt(sensitivity)-(1-sensitivity))/n
        reportJSON['sensitivity'] = {'value': sensitivity, 'CI': round(abs(sensitivityCI),4), 'n': n}
    except Exception as e:
        print(f"An error occurred while calculating metrics: {str(e)}")

    return reportJSON

# Cell 6 - Define model loading function

In [6]:
def load_model():
    try:
        model = load_learner('models/multi-class-pg.pkl')
        post_log({"message": "Model loaded successfully", "status": "In Progress"})
        return model
    except Exception as e:
        post_log({"message": f"An error occurred while loading the model: {str(e)}", "status": "Failed"})
        exit(1)

# Cell 7 - Main function


In [7]:
def main():
    model = load_model()
    files = get_file_list(sas_url=sas_url)
    results = []
    
    if files:
        for file in files:
            if file.name.split("/")[0] not in ["covid", "nofinding", "pneumonia"]:
                post_log({"message": f"When looking for the data labels in the parent folder name of covid/nofinding/pneumonia, this file ({file.name}) was not correctly labeled", "status": "In Progress"})

            file_content = download_file(file.name, sas_url=sas_url)
            if file_content:                
                pred_class = getPred(model, file_content)
                if(pred_class == ""):
                    post_log({"message": f"The model did not return a prediction for the file ({file.name})", "status": "In Progress"})
                    continue
                    
                actual_class = file.name.split('/')[0]
                reducedResult = "nofinding" if pred_class in ["nofinding", "pneumonia"] else "covid"
                reducedActual = "nofinding" if actual_class in ["nofinding", "pneumonia"] else "covid"

                results.append({'blob.name':file.name, 'actual':reducedActual, 'prediction':reducedResult})

    rawReport = {}
    rawReport['report'] = generateReport(results)

    print(json.dumps(rawReport, indent=4))
    finalReport = {
        "json_data": rawReport,
        "name": "COVID-19 X-Ray Classification Report",
        "status": "Completed",
    }
    post_report(finalReport)

# Cell 8 - Run the main function

In [8]:
if __name__ == "__main__":
    post_log({"message": "Starting the COVID-19 X-Ray Classification Report"})
    main()

/Users/pgupta/Code/EscrowAI-Sample-Validation-Template/example/covid-notebook/.venv/lib/python3.9/site-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


{
    "report": {
        "accuracy": {
            "value": 0.9333333333333333,
            "CI": 0.1175,
            "n": 15
        },
        "specificity": {
            "value": 0.8,
            "CI": 0.0907,
            "n": 15
        },
        "sensitivity": {
            "value": 0.9333333333333333,
            "CI": 0.1175,
            "n": 15
        }
    }
}
